# Lesson 01: Reading FIX messages
**Level:** beginner

**Goal:** read any FIX message by eye and in code. You'll split it into fields, name the tags,
validate BodyLength and CheckSum, decode enums and repeating groups, and follow one order from
the bridge to the LP and back.

**Data:** `data/fix_logs/BRIDGE_LP_*.log`. Each line is `<time logged by the bridge> <IN|OUT> <raw FIX message>`,
where IN = received from the LP and OUT = sent to the LP. As in most log viewers, the invisible SOH separator
(ASCII 1) is shown as `|`.

In [1]:
import sys; sys.path.insert(0, "..")
import pandas as pd
from bridgelab import fix
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_colwidth", 120)

with open("../data/fix_logs/BRIDGE_LP_B.log") as f:
    lines = [next(f) for _ in range(8)]
for line in lines:
    print(line.rstrip()[:200])

20251015-11:29:58.500 OUT 8=FIX.4.4|9=73|35=A|49=BRIDGE01|56=LP_B|34=1|52=20251015-11:29:58.500|98=0|108=10|141=Y|10=117|
20251015-11:29:58.518 IN  8=FIX.4.4|9=73|35=A|49=LP_B|56=BRIDGE01|34=1|52=20251015-11:29:58.509|98=0|108=10|141=Y|10=126|
20251015-11:29:58.521 OUT 8=FIX.4.4|9=121|35=V|49=BRIDGE01|56=LP_B|34=2|52=20251015-11:29:58.521|262=MD-LP_B|263=1|264=2|265=0|146=3|55=EURUSD|55=GBPUSD|55=XAUUSD|10=227|
20251015-11:30:00.457 IN  8=FIX.4.4|9=218|35=W|49=LP_B|56=BRIDGE01|34=2|52=20251015-11:30:00.449|262=MD-LP_B-XAUUSD|55=XAUUSD|268=4|269=0|270=4085.33|271=200|290=1|269=0|270=4085.30|271=400|290=2|269=
20251015-11:30:00.644 IN  8=FIX.4.4|9=218|35=W|49=LP_B|56=BRIDGE01|34=3|52=20251015-11:30:00.634|262=MD-LP_B-XAUUSD|55=XAUUSD|268=4|269=0|270=4085.36|271=300|290=1|269=0|270=4085.33|271=600|290=2|269=
20251015-11:30:01.210 IN  8=FIX.4.4|9=234|35=W|49=LP_B|56=BRIDGE01|34=4|52=20251015-11:30:01.201|262=MD-LP_B-GBPUSD|55=GBPUSD|268=4|269=0|270=1.33475|271=2000000|290=1|269=0|270=1.334

## 1. Split a message into tag=value fields
A message is just text. Split on the separator, then split each piece on the first `=`.

In [2]:
log_time, direction, raw = lines[0][:21], lines[0][22:25].strip(), lines[0][26:].strip()
print("logged at:", log_time, "| direction:", direction)
print(raw)

fields = [part.split("=", 1) for part in raw.split("|") if part]
pd.DataFrame(fields, columns=["tag", "value"]).assign(name=lambda d: d.tag.astype(int).map(fix.TAGS))

logged at: 20251015-11:29:58.500 | direction: OUT
8=FIX.4.4|9=73|35=A|49=BRIDGE01|56=LP_B|34=1|52=20251015-11:29:58.500|98=0|108=10|141=Y|10=117|


,tag,value,name
0,8,FIX.4.4,BeginString
1,9,73,BodyLength
2,35,A,MsgType
3,49,BRIDGE01,SenderCompID
4,56,LP_B,TargetCompID
5,34,1,MsgSeqNum
6,52,20251015-11:29:58.500,SendingTime
7,98,0,EncryptMethod
8,108,10,HeartBtInt
9,141,Y,ResetSeqNumFlag


**How to read it:** `35=A` is a **Logon**, sent by `49=BRIDGE01` to `56=LP_B`. It is message number `34=1` in this
direction, it asks for a heartbeat every `108=10` seconds, and `141=Y` asks both sides to reset their sequence
numbers to 1 (normal at the start of the day).

## 2. Header, body and trailer
| Block | Tags (in order) | Rule |
|---|---|---|
| Header | `8` BeginString, `9` BodyLength, `35` MsgType, then `49`, `56`, `34`, `52` (+ optional `43`, `122`) | 8, 9 and 35 must be the first three |
| Body | message-specific | |
| Trailer | `10` CheckSum | always last |

## 3. Validate BodyLength (9) and CheckSum (10) by hand
* **BodyLength** = number of characters after `9=...|` up to and including the `|` before `10=`
* **CheckSum** = sum of the ASCII codes of every character before `10=` (with real SOH bytes), modulo 256, as 3 digits

In [3]:
SOH = "\x01"
msg = raw.replace("|", SOH)
start_body = msg.index(SOH, msg.index("9=")) + 1           # character after "9=73<SOH>"
start_trailer = msg.rindex(SOH + "10=") + 1                 # position of "10="
body = msg[start_body:start_trailer]
print("BodyLength declared:", msg.split(SOH)[1].split("=")[1], "| actual:", len(body))
print("CheckSum declared:  ", msg[start_trailer + 3:start_trailer + 6],
      "| actual:", f"{sum(msg[:start_trailer].encode()) % 256:03d}")

BodyLength declared: 73 | actual: 73
CheckSum declared:   117 | actual: 117


The library function `fix.validate()` does exactly this. If a single character changes, for example a price
edited by hand or a corrupted packet, the checksum no longer matches and the receiver must reject the message
(session-level `35=3 Reject`).

In [4]:
order_line = next(l for l in open("../data/fix_logs/BRIDGE_LP_B.log") if "35=D|" in l)
order_raw = order_line[26:].strip()
print(order_raw)
print("original :", fix.validate(order_raw))
tampered = order_raw.replace("|44=", "|44=9")               # someone changes the price
print("tampered :", fix.validate(tampered))

8=FIX.4.4|9=148|35=D|49=BRIDGE01|56=LP_B|34=6|52=20251015-11:30:34.547|11=BR700004-3|1=BRIDGE01|55=XAUUSD|54=2|60=20251015-11:30:34.547|38=490|40=2|44=4085.67|59=3|10=218|
original : {'body_length_declared': 148, 'body_length_actual': 148, 'checksum_declared': '218', 'checksum_actual': '218', 'body_length_ok': True, 'checksum_ok': True, 'valid': True}
tampered : {'body_length_declared': 148, 'body_length_actual': 149, 'checksum_declared': '218', 'checksum_actual': '019', 'body_length_ok': False, 'checksum_ok': False, 'valid': False}


## 4. Decode the message: tag names and enum values
Numbers like `54=2` or `59=3` mean nothing until you decode them. `fix.explain()` uses a small FIX 4.4
dictionary (see `bridgelab/fix.py`).

In [5]:
pd.DataFrame(fix.explain(order_raw))

,tag,name,value,meaning
0,8,BeginString,FIX.4.4,
1,9,BodyLength,148,
2,35,MsgType,D,NewOrderSingle
3,49,SenderCompID,BRIDGE01,
4,56,TargetCompID,LP_B,
5,34,MsgSeqNum,6,
6,52,SendingTime,20251015-11:30:34.547,
7,11,ClOrdID,BR700004-3,
8,1,Account,BRIDGE01,
9,55,Symbol,XAUUSD,


**In plain English:** the bridge sends LP_B a **NewOrderSingle** to **sell** (`54=2`) 538 ounces of **XAUUSD** as a
**limit** order (`40=2`) at `44=` the price LP_B quoted, **Immediate-Or-Cancel** (`59=3`): fill now or cancel.
`11=BR...` is the **ClOrdID**, the bridge's own reference that ties the LP's answers back to this order.

## 5. Build a message yourself
Building is the reverse: write the fields, then compute 9 and 10. `fix.build()` does it, so here is a
heartbeat and a test request.

In [6]:
hb = fix.build("0", "BRIDGE01", "LP_A", 1234, "20251015-12:00:00.000", [], sep="|")
tr = fix.build("1", "BRIDGE01", "LP_A", 1235, "20251015-12:00:05.000", [(112, "PING-1")], sep="|")
print(hb, fix.validate(hb)["valid"])
print(tr, fix.validate(tr)["valid"])

8=FIX.4.4|9=58|35=0|49=BRIDGE01|56=LP_A|34=1234|52=20251015-12:00:00.000|10=159| True
8=FIX.4.4|9=69|35=1|49=BRIDGE01|56=LP_A|34=1235|52=20251015-12:00:05.000|112=PING-1|10=006| True


## 6. Repeating groups: a market-data snapshot (35=W)
`268=4` (NoMDEntries) announces 4 entries. Each entry starts with `269` (0 = bid, 1 = offer), followed by
`270` price, `271` size and `290` level. Parsing must respect this structure, because the same tag appears many times.

In [7]:
md_raw = next(l for l in open("../data/fix_logs/BRIDGE_LP_B.log") if "35=W|" in l and "EURUSD" in l)[26:].strip()
print(md_raw)
m = fix.parse(md_raw)
book = pd.DataFrame(m["groups"]).rename(columns={269: "side", 270: "price", 271: "size", 290: "level"})
book["side"] = book.side.map({"0": "BID", "1": "OFFER"})
book

8=FIX.4.4|9=235|35=W|49=LP_B|56=BRIDGE01|34=10|52=20251015-11:30:03.445|262=MD-LP_B-EURUSD|55=EURUSD|268=4|269=0|270=1.16247|271=1500000|290=1|269=0|270=1.16244|271=3000000|290=2|269=1|270=1.16253|271=1500000|290=1|269=1|270=1.16256|271=3000000|290=2|10=003|


,side,price,size,level
0,BID,1.16247,1500000,1
1,BID,1.16244,3000000,2
2,OFFER,1.16253,1500000,1
3,OFFER,1.16256,3000000,2


In [8]:
best_bid = float(book[(book.side == "BID") & (book.level == "1")].price.iloc[0])
best_ask = float(book[(book.side == "OFFER") & (book.level == "1")].price.iloc[0])
print(f"LP_B EURUSD top of book {best_bid} / {best_ask} -> spread = {(best_ask - best_bid) / 0.0001:.1f} pips")

LP_B EURUSD top of book 1.16247 / 1.16253 -> spread = 0.6 pips


## 7. Message types in a whole session log
Load the full LP_B log and count what's in it. Market data dominates; orders and execution reports are a
small fraction.

In [9]:
rows = []
for log_time, direction, raw_msg in fix.read_log("../data/fix_logs/BRIDGE_LP_B.log"):
    f = fix.parse(raw_msg)
    rows.append(dict(log_time=log_time, direction=direction, msg_type=f[35], name=fix.MSG_TYPES.get(f[35]),
                     seq=int(f[34]), raw=raw_msg))
log = pd.DataFrame(rows)
log.groupby(["direction", "msg_type", "name"]).size().rename("messages").reset_index()

,direction,msg_type,name,messages
0,IN,4,SequenceReset,2
1,IN,8,ExecutionReport,168
2,IN,A,Logon,1
3,IN,W,MarketDataSnapshotFullRefresh,22149
4,OUT,0,Heartbeat,657
5,OUT,2,ResendRequest,1
6,OUT,A,Logon,1
7,OUT,D,NewOrderSingle,142
8,OUT,V,MarketDataRequest,1


## 8. Follow one order end to end (D → 8 → re-route → 8)
Pick a client order that was **rejected by one LP and re-routed** to another. ClOrdIDs are `BR<platform order id>-<attempt>`,
so attempt `-1` and attempt `-2` belong to the same client order.

In [10]:
all_rows = []
for lp in ["LP_A", "LP_B", "LP_C"]:
    for log_time, direction, raw_msg in fix.read_log(f"../data/fix_logs/BRIDGE_{lp}.log"):
        if "35=D|" in raw_msg or "35=8|" in raw_msg:
            f = fix.parse(raw_msg)
            all_rows.append(dict(lp=lp, log_time=log_time, direction=direction, msg=fix.MSG_TYPES[f[35]],
                                 cl_ord_id=f.get(11), exec_type=fix.ENUMS[150].get(f.get(150), ""),
                                 side=fix.ENUMS[54][f[54]], qty=f.get(38), price=f.get(44) or f.get(31),
                                 last_qty=f.get(32), text=f.get(58, "")))
orders_fix = pd.DataFrame(all_rows)
orders_fix["client_order"] = orders_fix.cl_ord_id.str.extract(r"BR(\d+)-")[0]
rerouted = orders_fix[(orders_fix.exec_type == "Rejected")].client_order
example = orders_fix[orders_fix.client_order.isin(rerouted)].client_order.iloc[0]
orders_fix[orders_fix.client_order == example].sort_values("log_time").drop(columns="client_order")

,lp,log_time,direction,msg,cl_ord_id,exec_type,side,qty,price,last_qty,text
1157,LP_C,20251015-11:34:24.919,OUT,NewOrderSingle,BR700056-1,,Sell,0.37,4086.48,NaN,
1158,LP_C,20251015-11:34:25.063,IN,ExecutionReport,BR700056-1,Rejected,Sell,0.37,NaN,NaN,Off quotes
23,LP_A,20251015-11:34:25.064,OUT,NewOrderSingle,BR700056-2,,Sell,37,4086.45,NaN,
24,LP_A,20251015-11:34:25.071,IN,ExecutionReport,BR700056-2,Trade,Sell,37,4086.45,37,


**Reading the timeline:**
1. `OUT D` the bridge sends the order to the best-priced LP (attempt `-1`).
2. `IN 8 Rejected` the LP rejects it (`58=` gives the reason: *Off quotes*, *Price moved*…).
3. `OUT D` the bridge immediately re-routes to the next best LP (attempt `-2`).
4. `IN 8 Trade` that LP fills it: `31` LastPx is the price, `32` LastQty is the quantity.

Time between the D and its answer = the LP's **round-trip + hold time**. You'll measure it for every order in Lesson 05.

## 9. Partial fills
When the order is bigger than the LP's available size, the LP fills part of it (`39=1` PartiallyFilled) and,
because the order is IOC, cancels the rest (`150=4` Canceled). The bridge then routes the remainder.

In [11]:
partial = orders_fix[orders_fix.text.str.contains("IOC remainder")].client_order.iloc[0]
orders_fix[orders_fix.client_order == partial].sort_values("log_time").drop(columns="client_order")

,lp,log_time,direction,msg,cl_ord_id,exec_type,side,qty,price,last_qty,text
1133,LP_C,20251015-11:30:34.405,OUT,NewOrderSingle,BR700004-1,,Sell,9.4,4085.76,NaN,
1134,LP_C,20251015-11:30:34.536,IN,ExecutionReport,BR700004-1,Trade,Sell,9.4,4085.76,1.5,
0,LP_A,20251015-11:30:34.537,OUT,NewOrderSingle,BR700004-2,,Sell,790,4085.73,NaN,
1135,LP_C,20251015-11:30:34.537,IN,ExecutionReport,BR700004-1,Canceled,Sell,9.4,NaN,NaN,IOC remainder cancelled
1,LP_A,20251015-11:30:34.546,IN,ExecutionReport,BR700004-2,Trade,Sell,790,4085.73,300,
2,LP_A,20251015-11:30:34.547,IN,ExecutionReport,BR700004-2,Canceled,Sell,790,NaN,NaN,IOC remainder cancelled
823,LP_B,20251015-11:30:34.547,OUT,NewOrderSingle,BR700004-3,,Sell,490,4085.67,NaN,
824,LP_B,20251015-11:30:34.585,IN,ExecutionReport,BR700004-3,Trade,Sell,490,4085.67,200,
825,LP_B,20251015-11:30:34.586,IN,ExecutionReport,BR700004-3,Canceled,Sell,490,NaN,NaN,IOC remainder cancelled


## Cheat sheet
| Tag | Name | Typical values |
|---|---|---|
| 35 | MsgType | A Logon, 0 Heartbeat, 1 TestRequest, 2 ResendRequest, 4 SequenceReset, 5 Logout, V MD request, W MD snapshot, D NewOrderSingle, 8 ExecutionReport |
| 34 / 52 | MsgSeqNum / SendingTime | increasing per direction / sender's clock |
| 11 / 37 / 17 | ClOrdID / OrderID / ExecID | bridge's id / LP's id / unique id of each execution event |
| 54 / 38 / 44 / 40 / 59 | Side / Qty / Price / OrdType / TimeInForce | 1 buy, 2 sell / units / limit price / 2 limit / 3 IOC |
| 150 / 39 | ExecType / OrdStatus | 0 new, F trade, 4 canceled, 8 rejected / 1 partial, 2 filled, 8 rejected |
| 31 / 32 / 14 / 151 / 6 | LastPx / LastQty / CumQty / LeavesQty / AvgPx | per-fill and cumulative quantities |
| 58 | Text | reject reason |
| 43 / 122 | PossDupFlag / OrigSendingTime | Y on re-sent messages (Lesson 02) |

**Next:** Lesson 02, the session layer (heartbeats, sequence numbers, gaps, disconnections).